In [1]:
# -*- coding: utf-8 -*-
import numpy as np
import os
import h5py
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree
from tqdm import tqdm
from joblib import Parallel, delayed
import pymc as pm
import arviz as az

In [2]:
# 0) read TNG300-1 groupcat_99
base_path = "/z/fub_huang/TNG300-1/groupcat_99"
file_template = "fof_subhalo_tab_099.{}.hdf5"

halo_data = {
    'Group/GroupPos': [],
    'Group/GroupVel': [],
    'Group/Group_M_Crit200': [],
    'Group/Group_R_Crit200': []
}
subhalo_data = {
    'Subhalo/SubhaloPos': [],
    'Subhalo/SubhaloVel': [],
    'Subhalo/SubhaloMass': [],
    'Subhalo/SubhaloGrNr':[]
}

for i in tqdm(range(600), desc="Loading groupcat_99"):
    fp = os.path.join(base_path, file_template.format(i))
    if not os.path.exists(fp):
        continue
    with h5py.File(fp, 'r') as f:
        for k in halo_data:
            if k in f:
                halo_data[k].append(f[k][:])
        for k in subhalo_data:
            if k in f:
                subhalo_data[k].append(f[k][:])

for k in halo_data:
    halo_data[k] = np.concatenate(halo_data[k]) if halo_data[k] else np.array([])
for k in subhalo_data:
    subhalo_data[k] = np.concatenate(subhalo_data[k]) if subhalo_data[k] else np.array([])

print("Loaded:",
      {k: v.shape for k, v in halo_data.items()},
      {k: v.shape for k, v in subhalo_data.items()}, sep="\n")

Loading groupcat_99: 100%|████████████████████████████████████████████████████████████| 600/600 [00:45<00:00, 13.28it/s]


Loaded:
{'Group/GroupPos': (17625892, 3), 'Group/GroupVel': (17625892, 3), 'Group/Group_M_Crit200': (17625892,), 'Group/Group_R_Crit200': (17625892,)}
{'Subhalo/SubhaloPos': (14485709, 3), 'Subhalo/SubhaloVel': (14485709, 3), 'Subhalo/SubhaloMass': (14485709,), 'Subhalo/SubhaloGrNr': (14485709,)}


In [3]:
# 1) units
h = 0.704
z = 0.0
a = 1.0 / (1.0 + z)

# [UNIT] positions: ckpc/h -> Mpc(phys)
halo_pos = np.array(halo_data['Group/GroupPos']) * a / (h * 1000.0)            # Mpc
subhalo_pos = np.array(subhalo_data['Subhalo/SubhaloPos']) * a / (h * 1000.0)  # Mpc

# [UNIT] velocities: GroupVel km/s/a -> km/s ; SubhaloVel already km/s
halo_vel = np.array(halo_data['Group/GroupVel']) / a                            # km/s
subhalo_vel = np.array(subhalo_data['Subhalo/SubhaloVel'])                      # km/s

# [UNIT] masses:
#   halo catalog masses: 1e10 Msun/h
#   subhalo masses:      1e12 Msun
# halo_mass_cat = np.array(halo_data['Group/Group_M_Crit200']) / (h)                   # 1e10 Msun/h
# subhalo_mass = subhalo_data['Subhalo/SubhaloMass'] / h                # 1e10 Msun

# constant
G = 4.302e-6 / 1000.0     # Mpc (km/s)^2 / Msun
C_proj = 16.0 / np.pi     # PME constant

# Halo mass: Group_M_Crit200 [1e10 Msun/h] → Msun
halo_mass_cat = np.array(halo_data['Group/Group_M_Crit200']) * 1e10 / h   # Msun

# Subhalo mass: SubhaloMass [1e10 Msun/h] → Msun
subhalo_mass  = np.array(subhalo_data['Subhalo/SubhaloMass']) * 1e10 / h  # Msun

halo_mass_cat_1e12 = halo_mass_cat / 1e12
subhalo_mass_1e12  = subhalo_mass / 1e12

print("Halo mass range [1e12 Msun]:", halo_mass_cat_1e12.min(), halo_mass_cat_1e12.max())
print("Subhalo mass range [1e12 Msun]:", subhalo_mass_1e12.min(), subhalo_mass_1e12.max())

Halo mass range [1e12 Msun]: 0.0 1477.7615
Subhalo mass range [1e12 Msun]: 9.1709124e-05 1822.9347


In [4]:
min_subhalo_mass = subhalo_mass.min()
print("Minimum subhalo mass [Msun]:", min_subhalo_mass)
print("Minimum subhalo mass [1e12 Msun]:", min_subhalo_mass / 1e12)

Minimum subhalo mass [Msun]: 91709120.0
Minimum subhalo mass [1e12 Msun]: 9.1709124e-05


In [5]:
# -----------------------------
# Cosmology / constants
# -----------------------------
z = 0.0
a = 1.0              # scale factor at z=0 -> 1

# TNG300 box：205 Mpc/h（comoving）。z=0 to Mpc：
Lbox_Mpch = 205.0                 # Mpc/h
Lbox_Mpc  = (Lbox_Mpch / h) * a   # Mpc (physical)

# Sandage(1980) ：R0 = (8 G M t_U^2 / pi^2)^(1/3)
# G [Mpc^3 / (Msun Gyr^2)], M [Msun], t_U [Gyr], R0 [Mpc]
t_U = 13.8                        # Gyr，
G   = 4.498638185699749e-15       # Mpc^3 / (Msun Gyr^2)

# -----------------------------
# Load & unit conversions
# -----------------------------
# Group/GroupPos: ckpc/h -> Mpc(physical)
halo_pos = np.array(halo_data['Group/GroupPos']) * a / (h * 1000.0)  # Mpc

# Group/Group_M_Crit200: 1e10 Msun/h -> Msun
halo_mass = np.array(halo_data['Group/Group_M_Crit200']) * 1e10 / h  # Msun

N = len(halo_pos)
print(f"N groups under test: {N}")

# -----------------------------
# Turnaround radius R0 (in Mpc)
# -----------------------------
R0 = ((8.0 * G * halo_mass * (t_U**2)) / (np.pi**2))**(1.0/3.0)  # Mpc

# -----------------------------
# KDTree all halo (condition)
# -----------------------------
halo_pos = halo_pos % Lbox_Mpc   # wrap to box range
tree = cKDTree(halo_pos, boxsize=Lbox_Mpc)

# -----------------------------
# isolate check
# -----------------------------
mass_ratio_cut = 0.1  # only ≥ 0.1 * M_self neighbora
isolated_masscut = np.ones(N, dtype=bool)

for i in tqdm(range(N), desc=f"Checking isolation (mass_ratio_cut={mass_ratio_cut})"):
    M_self = halo_mass[i]
    r_search = R0[i]

    # find neighbors
    neigh = tree.query_ball_point(halo_pos[i], r=float(r_search))
    if len(neigh) <= 1:
        continue  # no neighbor = isolate

    # check large neighbor
    has_big = False
    for j in neigh:
        if j == i:
            continue
        if halo_mass[j] >= mass_ratio_cut * M_self:
            has_big = True
            break

    if has_big:
        isolated_masscut[i] = False

print(f"[Mass-ratio cut {mass_ratio_cut}] Isolated fraction: "
      f"{isolated_masscut.sum()}/{N} = {isolated_masscut.mean():.3f}")


# result
isolated_halos = {
    'positions_Mpc': halo_pos[isolated_masscut],
    'masses_Msun'  : halo_mass[isolated_masscut],
    'R0_Mpc'       : R0[isolated_masscut],
    #'d1_Mpc'       : d_nearest[isolated_masscut],  
}

N groups under test: 17625892


Checking isolation (mass_ratio_cut=0.1): 100%|███████████████████████████| 17625892/17625892 [03:11<00:00, 92193.25it/s]


[Mass-ratio cut 0.1] Isolated fraction: 16773368/17625892 = 0.952


In [6]:
# ---- parameter ----
f_R = 1.0            # subhalo searching = f_R * R0[i]
min_sub = 10          # at least min_sub subhalos
chunk_size = 10000   # size per time

# ---- data（isolated, halo_pos, R0, subhalo_pos）----
iso_idx = np.where(isolated_masscut)[0]      # isolated halo index
pos_iso = halo_pos[iso_idx]          # pos (Mpc)
r_cut = f_R * R0[iso_idx]            # search radius (Mpc)

# ---- create subhalo KDTree ----
sub_tree = cKDTree(subhalo_pos % Lbox_Mpc, boxsize=Lbox_Mpc)

# ---- check nuber of subhalos----
counts = np.zeros(len(iso_idx), dtype=np.int32)

for start in tqdm(range(0, len(iso_idx), chunk_size), desc="Counting subhalos around isolated halos"):
    end = min(start + chunk_size, len(iso_idx))
    # get list of lists
    neigh_lists = sub_tree.query_ball_point(pos_iso[start:end], r=r_cut[start:end])
    counts[start:end] = [len(lst) for lst in neigh_lists]

# ---- keep subhalo ≥ min_sub,print numbers ----
mask_ge = counts >= min_sub
num_kept = int(mask_ge.sum())
print(f"Isolated halos with >= {min_sub} subhalos (within {f_R} * R0): {num_kept}")

# ---- get mass ----
kept_isolated_indices = iso_idx[mask_ge]
kept_masses = halo_mass[kept_isolated_indices]   # in Msun

# delete (<=0)
kept_masses = kept_masses[kept_masses > 0]

len(kept_masses)

Counting subhalos around isolated halos: 100%|██████████████████████████████████████| 1678/1678 [00:57<00:00, 29.22it/s]

Isolated halos with >= 10 subhalos (within 1.0 * R0): 73694


73694

In [7]:
# -------------------------------
# Constants
# -------------------------------
G_cosmo = 4.498638185699749e-15   # Mpc^3 / (Msun Gyr^2)
G_dyn   = 4.302e-9                # Mpc (km/s)^2 / Msun
C_rad   = 16/np.pi                # TME / PME coefficient

# -------------------------------
# Box size & observer position (mock observer)
# -------------------------------
h = 0.704
Lbox_Mpch = 205.0              # Mpc/h  (TNG300 comoving)
Lbox_Mpc  = Lbox_Mpch / h      # physical Mpc
observer_pos = np.array([Lbox_Mpc/2.0]*3)   # observer at box center (Mpc)

# -------------------------------
# Halo catalog data (physical units)
# -------------------------------
halo_pos_full      = np.array(halo_data['Group/GroupPos']) * a / (h * 1000.0)   # Mpc
halo_vel_full      = np.array(halo_data['Group/GroupVel']) / a                  # km/s
halo_mass_cat_full = np.array(halo_data['Group/Group_M_Crit200']) * 1e10 / h    # Msun

R0_full = ((8.0 * G_cosmo * halo_mass_cat_full * (t_U**2)) / (np.pi**2))**(1/3) # Mpc

# R200_full = np.array(halo_data['Group/Group_R_Crit200']) * a / (h * 1000.0)  # Mpc

# # approximate Rvir = 1.2 * R200
# R0_full = 1.2 * R200_full

assert len(halo_pos_full) == len(halo_vel_full) == len(halo_mass_cat_full) == len(R0_full)

# -------------------------------
# Assume the subhalo KDTree has already been built:
# sub_tree = cKDTree(subhalo_pos)   # subhalo_pos in Mpc
# -------------------------------
def process_isolated_halo(i, use_energy_filter=True):
    """
    True mock-observer 2D PME:

    - Observer located at box center (observer_pos)
    - Use observer→host and observer→sub directions to compute sky angle θ
    - R_proj = D_host * θ  (exactly angle × distance)
    - v_los = (v_sub - v_host) · n_hat_obs→host
    - Only keep halos with exactly 31 bound satellites (M81-like)
    """

    # -------------------------------
    # 0) host basic properties
    # -------------------------------
    m_cat = halo_mass_cat_full[i]     # Msun
    if m_cat <= 0:
        return None

    host_pos = halo_pos_full[i] % Lbox_Mpc
    host_vel = halo_vel_full[i]       # km/s
    R0       = R0_full[i]             # Mpc

    # -------------------------------
    # 1) search subhalos within R0 in 3D
    # -------------------------------
    neigh = sub_tree.query_ball_point(host_pos, r=float(R0))
    if len(neigh) < 3:
        return None

    # keep all quantities aligned to the same indices
    sub_pos  = subhalo_pos[neigh]     # (N,3)
    sub_vel  = subhalo_vel[neigh]     # (N,3)
    sub_mass = subhalo_mass[neigh]    # (N,)

    # -------------------------------
    # 2) true 3D shell filtering
    # -------------------------------
    vec_host_sub = sub_pos - host_pos          # (N,3)
    r_3d = np.linalg.norm(vec_host_sub, axis=1)

    r_eps = 1e-6
    ok_shell = (r_3d > r_eps) & np.isfinite(r_3d)
    if not np.any(ok_shell):
        return None

    vec_host_sub = vec_host_sub[ok_shell]
    r_3d         = r_3d[ok_shell]
    sub_pos      = sub_pos[ok_shell]
    sub_vel      = sub_vel[ok_shell]
    sub_mass     = sub_mass[ok_shell]

    # -------------------------------
    # 3) observer → host direction
    # -------------------------------
    vec_obs_host = host_pos - observer_pos          # (3,)
    D_host = np.linalg.norm(vec_obs_host)           # Mpc
    if not np.isfinite(D_host) or D_host <= 0:
        return None

    u_host = vec_obs_host / D_host                  # (3,)
    n_hat  = u_host                                # LOS direction

    # -------------------------------
    # 4) observer → sub direction
    # -------------------------------
    vec_obs_sub = sub_pos - observer_pos            # (N,3)
    D_sub = np.linalg.norm(vec_obs_sub, axis=1)     # (N,)

    good = (D_sub > 0) & np.isfinite(D_sub)
    if not np.any(good):
        return None

    vec_host_sub = vec_host_sub[good]
    r_3d         = r_3d[good]
    sub_pos      = sub_pos[good]
    sub_vel      = sub_vel[good]
    sub_mass     = sub_mass[good]
    vec_obs_sub  = vec_obs_sub[good]
    D_sub        = D_sub[good]

    # -------------------------------
    # 5) angular separation → projected radius R_proj
    # -------------------------------
    u_sub = vec_obs_sub / D_sub[:, None]            # (N,3)

    cos_theta = np.sum(u_sub * u_host[None, :], axis=1)
    cos_theta = np.clip(cos_theta, -1.0, 1.0)
    theta = np.arccos(cos_theta)                   # radians

    R_proj = D_host * theta                        # Mpc

    ok_R = (R_proj > r_eps) & np.isfinite(R_proj)
    if not np.any(ok_R):
        return None

    vec_host_sub = vec_host_sub[ok_R]
    r_3d         = r_3d[ok_R]
    sub_pos      = sub_pos[ok_R]
    sub_vel      = sub_vel[ok_R]
    sub_mass     = sub_mass[ok_R]
    R_proj       = R_proj[ok_R]

    # -------------------------------
    # 6) line-of-sight velocity
    # -------------------------------
    rel_vel = sub_vel - host_vel                   # (N,3)
    v_los   = rel_vel @ n_hat                      # (N,)

    # -------------------------------
    # 7) energy filtering (final bound selection)
    # -------------------------------
    if use_energy_filter:
        v2 = np.sum(rel_vel**2, axis=1)
        E  = 0.5 * v2 - (G_dyn * m_cat / r_3d)
        bound_mask = np.isfinite(E) & (E < 0)
    else:
        bound_mask = np.isfinite(v_los)

    # need at least X bound satellites
    X = 31

    if np.count_nonzero(bound_mask) < X:
        return None

    # final synchronized trimming
    v_los_b        = v_los[bound_mask]
    R_proj_b       = R_proj[bound_mask]
    rel_vel_b      = rel_vel[bound_mask]
    vec_host_sub_b = vec_host_sub[bound_mask]
    sub_mass_b     = sub_mass[bound_mask]

    # -------------------------------
    # 8) enforce exactly X bound satellites
    # -------------------------------
    n_bound = len(sub_mass_b)

    # if more than X → keep the most massive X
    if n_bound > X:
        idx_sort = np.argsort(sub_mass_b)[::-1]
        idx_keep = idx_sort[:X]

        v_los_b        = v_los_b[idx_keep]
        R_proj_b       = R_proj_b[idx_keep]
        rel_vel_b      = rel_vel_b[idx_keep]
        vec_host_sub_b = vec_host_sub_b[idx_keep]
        sub_mass_b     = sub_mass_b[idx_keep]
        n_bound        = X

    # must be exactly X
    if n_bound != X:
        return None

    # -------------------------------
    # 9) PME mass (observational form)
    #    M_P = (16 / (π G)) (1/N) Σ (v_los^2 R_proj)
    # -------------------------------
    M_tme = (C_rad / (G_dyn * n_bound)) * np.sum((v_los_b**2) * R_proj_b)

    # -------------------------------
    # 10) statistics: sigma_v, v_mean
    # -------------------------------
    v_mean  = float(np.mean(v_los_b))   # km/s
    sigma_v = float(np.sqrt(np.mean((v_los_b - v_mean)**2)))

    # optional internal geometry quantity
    proj_along = np.dot(vec_host_sub_b, n_hat)
    vec_perp   = vec_host_sub_b - proj_along[:, None] * n_hat[None, :]
    unit_perp  = vec_perp / R_proj_b[:, None]
    R_vec      = np.mean(unit_perp, axis=0)
    R_mag      = float(np.linalg.norm(R_vec))

    return i, M_tme, m_cat, n_bound, R_mag, sigma_v, v_mean


# -------------------------------
# Parallel computation + progress bar
# -------------------------------
results = Parallel(n_jobs=8, backend="threading")(
    delayed(process_isolated_halo)(int(i))
    for i in tqdm(kept_isolated_indices, desc="Mock-observer PME for isolated halos")
)

# remove None
results = [r for r in results if r is not None]
halo_ids, tme_masses, true_masses, n_subs, R_mags, sigma_v, v_means = map(np.array, zip(*results))

print(f"Computed mock-observer PME+features for {len(results)} isolated halos.")
print("Saved fields: halo_ids, tme_masses, true_masses, n_subs, R_mags, sigma_v, v_means")

assert np.all(n_subs == X)
print("All selected halos have exactly 31 bound satellites.")

Mock-observer PME for isolated halos: 100%|█████████████████████████████████████| 73694/73694 [00:23<00:00, 3185.23it/s]


Computed mock-observer PME+features for 13248 isolated halos.
Saved fields: halo_ids, tme_masses, true_masses, n_subs, R_mags, sigma_v, v_means


NameError: name 'X' is not defined

In [ ]:
np.savez("halo_features_2D_31-200.npz",
         halo_ids=halo_ids,
         tme_masses=tme_masses,
         true_masses=true_masses,
         n_subs=n_subs_R0,
         R_mags=R_mags,
         sigma_v=sigma_v,
         v_means=v_means)

print("Saved halo_features_2D.npz with keys:",
      ["halo_ids", "tme_masses", "true_masses", "n_subs", "R_mags", "sigma_v", "v_means"])

In [ ]:
pme = np.array(tme_masses)
true = np.array(true_masses)

logT = np.log10(true)
logP = np.log10(pme)

plt.figure(figsize=(8,6))

# use hexbin（density plot）
hb = plt.hexbin(logT, logP, gridsize=100, cmap='viridis', bins='log')  
plt.colorbar(hb, label='Counts')

# y = x line
plt.plot([6,17], [6,17], 'r--', label=r'$y=x$')

plt.xlabel(r'$\log_{10} M_{\rm true}\ (M_\odot)$')
plt.ylabel(r'$\log_{10} M_{\rm PME}\ (M_\odot)$')
plt.gca().set_aspect('equal', adjustable='box')
plt.legend()
plt.xlim(11,17)
plt.ylim(11,17)
plt.tight_layout()
plt.show()

len(logT)